# CNN + DFE equalizer on the PYNQ-Z2

This notebook runs the deployed equalizer (8-bit CNN with two decision-feedback taps, 200 MHz) on the board and
compares it with the golden integer model.

**Needs, in this folder:** `cnn_eq.bit`, `cnn_eq.hwh`, `board_vectors.npz`, `cnn_eq_pynq.py`.

**Expected:** all 4M outputs bit-exact with the golden model, **2 bit errors in 3,997,440 bits**
(BER 5.0e-7, the number in the report), and a core throughput close to 97.5 MSym/s.

In [ ]:
import json, numpy as np
from cnn_eq_pynq import Equalizer, run_test
import pynq
print('PYNQ', pynq.__version__)

## 1. Load the overlay and check the core

In [ ]:
v = np.load('board_vectors.npz')
eq = Equalizer()                       # programs the PL; PYNQ sets FCLK0 = 100 MHz and FCLK1 = 200 MHz from the .hwh
print({k: hex(x) if k != 'NPARAM' else x for k, x in eq.check(v['shape_word'], len(v['params'])).items()})

## 2. Load the 336 parameters and run one frame (200,000 symbols)

In [ ]:
eq.load_params(v['params'])
eq.soft_reset()
n_sym = int(v['n_sym'])
inb = pynq.allocate(shape=(v['x'].shape[1],), dtype=np.int8)
outb = pynq.allocate(shape=(n_sym,), dtype=np.int16)
inb[:] = v['x'][0]; inb.flush()
outb[:] = 0; outb.flush()
eq.run_frame(inb, outb)
outb.invalidate()
out = np.array(outb)
print('mismatches vs golden:', int(np.count_nonzero(out != v['u_gold'][0])), 'of', n_sym)
print('counters:', eq.counters())

Soft outputs of the first frame, split by the transmitted bit: the two classes should be cleanly separated by zero.

In [ ]:
import matplotlib.pyplot as plt
bits = np.unpackbits(v['bits_true'][0])[:n_sym].astype(bool)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(out[~bits], bins=200, alpha=0.7, label='sent 0')
ax.hist(out[bits], bins=200, alpha=0.7, label='sent 1')
ax.axvline(0, color='k', lw=1); ax.set_yscale('log'); ax.set_xlabel('equalizer output u (int16, from the FPGA)')
ax.legend(); plt.tight_layout(); plt.savefig('board_histogram.png', dpi=150); plt.show()

## 3. Full test: all 20 frames, bit-exactness, BER and throughput (writes `board_results.json`)

In [ ]:
res = run_test(repeats=3)
print(json.dumps(res, indent=1))

## 4. Send back
`board_results.json` and `board_histogram.png` (in this folder). If anything fails, also the full output of the
cells above.